# Задание 3. Восстановление фильтров KaiNet

Нужно восстановить неизвестный фильтр 3×3 и порядок трёх фильтров по двум известным ядрам и 1000 парам «изображение — результат».

Все вычисления приведены в этом блокноте; внешний Python-скрипт не требуется. Зависимости: numpy, scipy, Pillow.

Для повторного запуска скачайте [архив задания](https://disk.yandex.ru/d/ECLHqVyX-F5hRA) и распакуйте PNG, TXT и algos.csv в папку `data` рядом с блокнотом. Также поддерживается запуск из корня проекта или папки `lab2_pytorch`.


In [1]:
from pathlib import Path
from itertools import permutations
import numpy as np
from PIL import Image
from scipy.ndimage import correlate

locations = [Path('data'), Path('task3/data'), Path('lab2_pytorch/task3/data')]
DATA = next((p for p in locations if (p / 'algos.csv').exists()), None)
assert DATA is not None, 'Распакуйте архив в data рядом с блокнотом'

a, b = np.loadtxt(DATA / 'algos.csv', delimiter=',').reshape(2, 3, 3)
paths = sorted(DATA.glob('*.png'), key=lambda p: int(p.stem))
assert len(paths) == 1000
assert all(p.with_suffix('.txt').exists() for p in paths)
print('Изображений:', len(paths))
print('Фильтр A:\n', a)
print('Фильтр B:\n', b)


Изображений: 1000
Фильтр A:
 [[-1.  -0.5  0. ]
 [-0.5  0.5  0.5]
 [ 0.   0.5  1. ]]
Фильтр B:
 [[0.0625 0.0625 0.0625]
 [0.0625 0.0625 0.0625]
 [0.0625 0.0625 0.0625]]


## Линейная система

При фиксированных порядке фильтров и обработке границ выход линейно зависит от девяти неизвестных коэффициентов X. Подставляем девять базисных ядер (одна единица, остальные нули) и записываем результаты в столбцы матрицы D. Тогда искомые коэффициенты удовлетворяют D @ x = y.

Перебираем шесть порядков и пять способов обработки границ. Для подбора используем первые пять изображений, для последующей проверки — все 1000, включая 995 не использованных при подборе. Операция `correlate` не переворачивает ядро и соответствует определению из условия. Входные пиксели остаются в диапазоне 0–255, вычисления выполняются в float64.


In [2]:
def apply_filters(image, filters, mode='constant'):
    result = np.asarray(image, dtype=np.float64)
    for kernel in filters:
        result = correlate(result, kernel, mode=mode, cval=0.0)
    return result

fit_pairs = [(np.asarray(Image.open(p), dtype=float), np.loadtxt(p.with_suffix('.txt')))
             for p in paths[:5]]
candidates = []
for mode in ['constant', 'reflect', 'mirror', 'nearest', 'wrap']:
    for order in permutations('ABX'):
        matrices, targets = [], []
        for image, target in fit_pairs:
            columns = []
            for basis in np.eye(9).reshape(9, 3, 3):
                kernels = {'A': a, 'B': b, 'X': basis}
                columns.append(apply_filters(image, [kernels[n] for n in order], mode).ravel())
            matrices.append(np.stack(columns, axis=1))
            targets.append(target.ravel())
        design, target = np.concatenate(matrices), np.concatenate(targets)
        coeffs, _, rank, _ = np.linalg.lstsq(design, target, rcond=None)
        mse = float(np.mean((design @ coeffs - target) ** 2))
        candidates.append(dict(order=''.join(order), mode=mode, mse=mse,
                               rank=int(rank), kernel=coeffs.reshape(3, 3).tolist()))
candidates.sort(key=lambda c: c['mse'])
best = candidates[0]
assert best['rank'] == 9 and best['mse'] < 1e-15
# Recover exact short decimal coefficients; validate this rounding on every pair.
missing = np.round(np.array(best['kernel']), 12)
kernels = {'A': a, 'B': b, 'X': missing}

for candidate in candidates[:6]:
    print(f"{candidate['order']}, {candidate['mode']}: MSE={candidate['mse']:.6g}, rank={candidate['rank']}")
print('Восстановленный фильтр X:\n', missing)


XBA, constant: MSE=1.33507e-27, rank=9
BXA, constant: MSE=3.50913e-27, rank=9
BAX, constant: MSE=37.7623, rank=9
XAB, constant: MSE=533.837, rank=9
ABX, constant: MSE=809.023, rank=9
AXB, constant: MSE=864.448, rank=9
Восстановленный фильтр X:
 [[0.125 0.25  0.125]
 [0.25  0.5   0.25 ]
 [0.125 0.25  0.125]]


## Проверка на всех изображениях

Лучшие порядки: X → B → A и B → X → A. Найденная обработка границ — нулевое дополнение на один пиксель перед каждым фильтром. Коэффициенты округлены до 12 знаков; проверим, что округление не изменило результат.


In [3]:
validation = {}
for order in ['XBA', 'BXA']:
    squared_error = 0.0
    pixel_count = 0
    max_error = 0.0
    for path in paths:
        target = np.loadtxt(path.with_suffix('.txt'))
        result = apply_filters(np.asarray(Image.open(path)), [kernels[n] for n in order])
        assert result.shape == target.shape
        error = result - target
        squared_error += float(np.sum(error ** 2))
        pixel_count += error.size
        max_error = max(max_error, float(np.max(np.abs(error))))
    validation[order] = dict(images=len(paths), pixels=pixel_count,
                             mse=squared_error / pixel_count, max_abs_error=max_error)
    assert max_error == 0.0, validation[order]

for order, metrics in validation.items():
    print(order, metrics)


XBA {'images': 1000, 'pixels': 4096000, 'mse': 0.0, 'max_abs_error': 0.0}
BXA {'images': 1000, 'pixels': 4096000, 'mse': 0.0, 'max_abs_error': 0.0}


## Почему два порядка дают одинаковый результат

B — разделимый фильтр с одномерными весами [1, 1, 1], X — с весами [1, 2, 1]. С точностью до множителей их одномерные операторы с нулевыми границами имеют вид T и T + I. Они коммутируют, как и соответствующие двумерные разделимые операторы. Поэтому перестановку B и X невозможно различить по выходным изображениям.

На всех 1000 парах для обоих порядков MSE и максимальная абсолютная ошибка равны нулю. Для отправки использован порядок X → B → A.


In [4]:
answer = np.stack([missing, b, a]).reshape(3, 9)
output = DATA.parent / 'reconstructed_algos.csv'
np.savetxt(output, answer, delimiter=',', fmt='%.12g')
assert np.array_equal(np.loadtxt(output, delimiter=','), answer)
print(output.read_text())


0.125,0.25,0.125,0.25,0.5,0.25,0.125,0.25,0.125
0.0625,0.0625,0.0625,0.0625,0.0625,0.0625,0.0625,0.0625,0.0625
-1,-0.5,0,-0.5,0.5,0.5,0,0.5,1



## Результат контеста

Файл `reconstructed_algos.csv` принят 28 сентября 2026 года: **OK — полное решение**.

[Посылка №166717004](https://contest.yandex.ru/contest/75233/run-report/166717004/).
